# Week 11 — Contextual Bandits and Off-Policy Evaluation · Workbook

This is a reproducible **reference workbook**. Many supplied scaffold or check
cells directly call `project.solutions`; executing them checks the reference
demonstration, not your implementation. They do not automatically switch modes
when run interactively or with papermill.

For your own attempt, save a copy, replace the relevant reference wrapper with
your code, and make sure the checks call that code. Predict a boundary case
before running it. For independently checked TODO exercises with no fallback,
start with [the progressive coding projects](../docs/INTERACTIVE_LEARNING.md).

**Teaching companion:** [Worked examples, learning bridges and self-checks](teaching_guide.md).


**A bounded first investigation:** Diagnose the log before trusting its estimate. Start with **Exercises 5, 6 and 7** and the
[guide’s prediction, visual diagnostic and changed-input check](teaching_guide.md#notebook-investigation-diagnose-the-log-before-trusting-its-estimate).
Save your own prediction before running the reference cells. If you implement an exercise,
replace its reference call and verify that its check invokes your function.


**Teaching companion:** [Worked examples, derivations, analogies, and self-checks](teaching_guide.md). Read the relevant example before attempting its exercise.

In [ ]:
# Standard preamble.
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # for common/
sys.path.insert(0, os.path.abspath('.'))    # for project/
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np
from project import solutions as sol
from project.env import SyntheticLeadBandit
from project.agents import (
    UCB1, KLUCB, BetaThompson, LinUCB, LinTS, EpsilonGreedyLogger, kl_bernoulli,
)
from project.ope import ips, snips, doubly_robust, switch_dr, RidgeRewardModel, true_policy_value
from project.train import softmax_policy, uniform_policy, greedy_oracle_policy

import matplotlib.pyplot as plt


## Exercise 1 — UCB1 bonus (★)

Compute the UCB1 exploration bonus  $\sqrt{2 \log t / n_a}$  for $t = 100$ and $n_a = 10$.

> **Hint** (try yourself first): one numpy call: `np.sqrt(2*np.log(t)/n_a)`.

In [ ]:
# Reference solution (filled in for end-to-end execution)
ans1 = sol.exercise_1_ucb_bonus(t=100, n_a=10)
assert abs(ans1 - np.sqrt(2*np.log(100)/10)) < 1e-9
print('E1 bonus =', ans1)


## Exercise 2 — KL-UCB upper bound (★)

Implement the KL-UCB upper-confidence solver: given $\hat\mu_a, n_a, t$, return
$\max\{q \in [\hat\mu_a, 1] : n_a \cdot \mathrm{KL}(\hat\mu_a, q) \le \log t + 3 \log\log t\}$
via bisection.

> **Hint**: the function $q \mapsto \mathrm{KL}(\hat\mu_a, q)$ is monotone increasing on $[\hat\mu_a, 1]$ — perfect for bisection.

In [ ]:
ub_a = sol.exercise_2_klucb_upper(0.4, n_a=20, t=200)
ub_b = sol.exercise_2_klucb_upper(0.4, n_a=200, t=200)
assert ub_a > 0.4 and ub_b > 0.4 and ub_b < ub_a
print('KL-UCB upper bound (n=20):', round(ub_a, 4))
print('KL-UCB upper bound (n=200):', round(ub_b, 4))


## Exercise 3 — Thompson Sampling regret (★★)

Run Bernoulli Thompson Sampling on a 3-armed bandit with means
$[0.2, 0.5, 0.8]$ for $T = 1000$ rounds. Report cumulative regret.
The Lai–Robbins lower bound is roughly
$(\Delta_1/\mathrm{KL}(0.2,0.8) + \Delta_2/\mathrm{KL}(0.5,0.8)) \log T$;
TS should be in the right order of magnitude.

> **Hint**: use `BetaThompson(K=3, seed=0)` from `project.agents`.

In [ ]:
reg = sol.exercise_3_thompson_regret(T=1000, seed=0)
print('TS cumulative regret:', round(reg, 2))
assert reg < 0.6 * 1000


## Exercise 4 — LinUCB on synthetic insurance (★★)

Instantiate `SyntheticLeadBandit(seed=0)` and `LinUCB(K=6, d=8, alpha=1.0)`,
roll out for $T = 1000$ rounds, and report cumulative regret.

> **Hint**: see the LinUCB pseudo-code in `notes.pdf` chapter 4; the project module already implements it.

In [ ]:
reg = sol.exercise_4_linucb_regret(T=1000, seed=0)
print('LinUCB regret:', round(reg, 2))
assert reg < 0.5 * 1000


## Exercise 5 — IPS and SNIPS for a uniform target (★★)

Collect $n=500$ logged samples from a 0.3-greedy logger; compute the IPS
and SNIPS estimates of the uniform target policy.

> **Hint**: the uniform target is just `np.full(K, 1/K)`. SNIPS divides by the sum of weights.

In [ ]:
v_ips, v_snips = sol.exercise_5_ips_snips_uniform(n=500, seed=0)
print('IPS  =', round(v_ips, 3))
print('SNIPS =', round(v_snips, 3))
# Uniform target has true value = mean over arms of mean reward; should be in [0, 1]
assert 0.0 <= v_snips <= 1.0


## Exercise 6 — Doubly Robust against softmax target (★★)

Fit a ridge reward model on separate synthetic uniform-logger rows, then evaluate the softmax-true
policy with DR. Compare to the true (on-policy) value.

> **Hint**: `RidgeRewardModel(K, d).fit(fit_log)` returns a callable `qhat(x) -> np.ndarray of shape (K,)`.

In [ ]:
v_dr = sol.exercise_6_dr_softmax(n=1200, seed=0)
env = SyntheticLeadBandit(seed=0)
v_true = true_policy_value(env, softmax_policy(env, beta=4.0), n=4000, seed=42)
print('DR    =', round(v_dr, 3))
print('truth =', round(v_true, 3))
assert abs(v_dr - v_true) < 0.25


## Exercise 7 — Effective sample size (★)

Compute the effective sample size of a weight vector. Sanity-check:
$\mathrm{ESS}(\mathbf{1}) = n$, $\mathrm{ESS}([10, 0, \ldots, 0]) \approx 1$.

> **Hint**: $\mathrm{ESS} = (\sum_i w_i)^2 / \sum_i w_i^2$.

In [ ]:
w = np.ones(100)
assert abs(sol.exercise_7_ess(w) - 100.0) < 1e-9
w2 = np.array([10.0] + [0.0] * 99)
print('ESS uniform:', sol.exercise_7_ess(w))
print('ESS spiked:', sol.exercise_7_ess(w2))


## Exercise 8 — SWITCH-DR threshold sweep (★★)

Vary $\tau \in \{1, 5, 10, 50, 1000\}$ for the oracle-greedy target. Plot
estimate vs.\ $\tau$ and identify the bias-variance trade-off.

> **Hint**: small $\tau$ → mostly direct method (biased, low variance); large $\tau$ → recovers DR.

In [ ]:
taus = [1.0, 5.0, 10.0, 50.0, 1000.0]
estimates = [sol.exercise_8_switch_dr(n=800, tau=tau, seed=0) for tau in taus]
for tau, est in zip(taus, estimates):
    print(f'tau={tau:7.1f}  V_SWITCH-DR = {est:.3f}')
assert np.isfinite(estimates).all()
fig, ax = plt.subplots(figsize=(6,3))
ax.semilogx(taus,estimates,'o-'); ax.set(xlabel='SWITCH threshold tau',ylabel='Estimate on one fixed synthetic log')
fig.tight_layout(); plt.show()
print('One threshold sweep shows sensitivity, not repeated-sampling bias or variance.')


## Exercise 9 — synthetic insurance OPE memo (★★★, open-ended)

For three counterfactual policies (uniform, softmax-true, oracle-greedy) compute:
- The on-policy ground truth (we have it because the env is synthetic).
- SNIPS and DR estimates from a 1500-row 0.3-greedy log.

Identify which estimator is more reliable for each policy and write a short
recommendation.

> **Hint**: Inspect the observed weights, ESS, held-out reward-model errors, and uncertainty. A peaky policy can have poor overlap, but neither the sign of SNIPS error nor the best estimator is guaranteed.

In [ ]:
truth, snips_est, dr_est = sol.exercise_9_synthetic_ope(n=1500, seed=0)
print(f'softmax-true on-policy truth = {truth:.3f}')
print(f'SNIPS estimate              = {snips_est:.3f}  (err = {abs(snips_est-truth):.3f})')
print(f'DR    estimate              = {dr_est:.3f}  (err = {abs(dr_est-truth):.3f})')
assert np.isfinite([truth,snips_est,dr_est]).all()
print('Compare actual errors; no estimator must win on every sample.')


## Wrap-up

You have:
- Inspected reference UCB1, KL-UCB, Thompson and LinUCB calculations.
- Built the synthetic insurance lead-broker bandit and trained LinUCB to sub-linear regret.
- Collected $\varepsilon$-greedy logs *with propensities*.
- Run IPS, SNIPS, DR, and SWITCH-DR off-policy evaluations.
- Checked ESS arithmetic; the statistical investigation below measures realized weight concentration.

Next week: ranking, two-sided matching, and fairness.

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 — UCB1 bonus (★)</summary>

> **Hint** (try yourself first): one numpy call: `np.sqrt(2*np.log(t)/n_a)`.

</details>

<details>
<summary>Exercise 2 — KL-UCB upper bound (★)</summary>

> **Hint**: the function $q \mapsto \mathrm{KL}(\hat\mu_a, q)$ is monotone increasing on $[\hat\mu_a, 1]$ — perfect for bisection.

</details>

<details>
<summary>Exercise 3 — Thompson Sampling regret (★★)</summary>

> **Hint**: use `BetaThompson(K=3, seed=0)` from `project.agents`.

</details>

<details>
<summary>Exercise 4 — LinUCB on synthetic insurance (★★)</summary>

> **Hint**: see the LinUCB pseudo-code in `notes.pdf` chapter 4; the project module already implements it.

</details>

<details>
<summary>Exercise 5 — IPS and SNIPS for a uniform target (★★)</summary>

> **Hint**: the uniform target is just `np.full(K, 1/K)`. SNIPS divides by the sum of weights.

</details>

<details>
<summary>Exercise 6 — Doubly Robust against softmax target (★★)</summary>

> **Hint**: `RidgeRewardModel(K, d).fit(fit_log)` returns a callable `qhat(x) -> np.ndarray of shape (K,)`.

</details>

<details>
<summary>Exercise 7 — Effective sample size (★)</summary>

> **Hint**: $\mathrm{ESS} = (\sum_i w_i)^2 / \sum_i w_i^2$.

</details>

<details>
<summary>Exercise 8 — SWITCH-DR threshold sweep (★★)</summary>

> **Hint**: small $\tau$ → mostly direct method (biased, low variance); large $\tau$ → recovers DR.

</details>

<details>
<summary>Exercise 9 — synthetic insurance OPE memo (★★★, open-ended)</summary>

> **Hint**: Inspect the observed weights, ESS, held-out reward-model errors, and uncertainty. A peaky policy can have poor overlap, but neither the sign of SNIPS error nor the best estimator is guaranteed.

</details>

After solving, change one input or assumption and predict the result before
running it. If a cell calls `project.solutions`, passing it verifies the
reference demonstration; test your own implementation separately.


## Statistical investigation — Observed weights and repeated-sampling error

**Goal.** Explain estimator error when rare logged actions carry most of a fixed target policy. This bounded reference investigation runs on CPU without downloads.
Write your prediction in a new Markdown cell before executing it.

**Predict.** Freeze a two-action target π=(0.1,0.9), a logger (0.98,0.02), and reward
means (0.2,0.8). Compute V(π)=0.74 before running. Predict how weight
concentration affects estimation across independent datasets of 200 rows.
Do not interpret ESS as a confidence interval or a population-support proof.

**Work a small case by hand.** Importance weighting cancels the logger in expectation:
E_b[wR]=sum_a b(a)[pi(a)/b(a)]q(a)=sum_a pi(a)q(a), when the target has
support and these probabilities are correct. The target value is 0.1×0.2+0.9×0.8=0.74. Observed action weights are
0.1/0.98=5/49≈0.1020 and 0.9/0.02=45. IPS averages wR and can exceed the binary reward range in a finite sample.
SNIPS divides by the random sum of weights; its bounded range does not make
it unbiased. In 200 logger draws the expected
rare-action count is 200×0.02=4, but zero occurs with probability
0.98^200≈0.0176. If all rows chose the common action, every observed weight
equals 5/49. Then ESS=(200w)²/(200w²)=200 even though no observed reward
comes from the action carrying 90% of target mass.

**Reading bridge:** [Probability and statistics lesson](../docs/statistics/07-importance-weighting.md).
These are original course examples informed by the textbook sections in the
[statistics source catalog](../docs/statistics/SOURCES.md); textbook PDFs are not required to run them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from common.plotting import RL_PALETTE, use_rl_style
use_rl_style()
from common.statistics import importance_diagnostics
rng_stats=np.random.default_rng(111)
pb=np.array([.98,.02]);pi=np.array([.1,.9]);reward_means=np.array([.2,.8]);truth=pi@reward_means
ips_samples=[];snips_samples=[];ess_samples=[];rare_action_counts=[]
for repetition in range(300):
    actions=rng_stats.choice(2,size=200,p=pb)
    reward=rng_stats.binomial(1,reward_means[actions])
    diagnostics=importance_diagnostics(pi[actions],pb[actions]);w=diagnostics['weights']
    ips_samples.append(np.mean(w*reward));snips_samples.append(np.dot(w,reward)/w.sum())
    ess_samples.append(diagnostics['effective_sample_size'])
    rare_action_counts.append(np.count_nonzero(actions==1))
print('Exact target value:',truth)
print('Datasets with no rare-action row:', np.count_nonzero(np.array(rare_action_counts)==0))
print('Such datasets have constant observed weights and ESS=200 despite missing the action carrying 90% of target mass.')
for label,est in [('IPS',ips_samples),('SNIPS',snips_samples)]:
    error=np.asarray(est)-truth
    print(label,'empirical bias',error.mean(),'RMSE',np.sqrt(np.mean(error**2)))
fig,axes=plt.subplots(1,2,figsize=(9,3.3))
shared_bins=np.linspace(min(min(ips_samples),min(snips_samples)),max(max(ips_samples),max(snips_samples)),32)
axes[0].hist(ips_samples,bins=shared_bins,alpha=.5,label='IPS')
axes[0].hist(snips_samples,bins=shared_bins,alpha=.5,label='SNIPS')
axes[0].axvline(truth,color=RL_PALETTE['red'],ls='--');axes[0].legend()
axes[0].set(xlabel='Estimated target-policy value',ylabel='Independent datasets')
axes[1].scatter(ess_samples,np.abs(np.array(snips_samples)-truth),s=10,alpha=.5)
axes[1].set(xlabel='Observed importance-weight ESS',ylabel='Absolute SNIPS error')
fig.tight_layout();plt.show()


**Interpret and transfer.** Notice the high-error points at ESS=200: when no rare action is observed, all
observed weights are equal. ESS then equals the row count even though this
sample contains no outcome for the action receiving 90% of target mass. This
is a concrete reason ESS cannot establish coverage or accuracy.

There is no universal ordering of estimator errors. The repeated-sampling
summary is conditional on this fixed logger, target, reward mechanism and IID
context design. An adaptive logger needs its own uncertainty argument. DR
helpers elsewhere in this workbook fit nuisance models on separate synthetic
uniform-logger samples.

**Transfer.** Make the logger probability of action 1 exactly zero. Explain
why additional action-0 rows cannot identify its target reward. Keep that
identification failure separate from a finite dataset that happens to contain
no action-1 row under a positive logger.


**Read your result.** Read the high-error points at ESS=200 as the all-common-action case. Their
SNIPS estimates collapse to a sample mean near 0.2. This concrete counterexample
explains why an apparently excellent observed ESS cannot establish accuracy or
population support. The printed Monte Carlo bias/RMSE summarizes 300 datasets.

<details><summary>Worked feedback for the changed assumption</summary>

If the logger's action-1 probability is exactly zero, its reward mean r₁ is
unidentified from this log. The target value is 0.02+0.9r₁, compatible with
any value from 0.02 to 0.92 for Bernoulli rewards. More action-0 rows cannot
close that range. Positive probability 0.02 instead makes identification
possible in principle while leaving rare-event uncertainty in finite samples.

</details>
